# Target shapes: a line, a curve, or a step?

When you know the response column, `target_shapes` helps choose how to
represent each numeric predictor. It compares the variance explained by a
straight line, a binned curve, and a single threshold. A strong nonlinear
shape can be useful even when Pearson correlation is small.

Use `profile(df, target="response")` to add the pass to the default
profile. With an explicit detector list, include `target_shapes`. The pass
uses all complete, finite predictor–target pairs, with at least 30 rows
and two distinct predictor values. It reports at most ten decisive shapes.

## The three comparisons

The linear score is squared Pearson correlation. For a binned curve,
observations are sorted by the predictor and assigned to at most 20
approximately equal-count bins, keeping ties together. Each bin predicts
its target mean. With total squared deviation `TSS`, bin residual sum of
squares `RSS`, sample size `n`, and bin budget `B`, the adjusted score is:

```text
curve_R2 = max(0, 1 - (RSS / TSS) * (n - 1) / (n - B))
```

The single-step score is the variance explained by two target means on
either side of the best split between distinct predictor values. The
reported `step_at` is the midpoint between the two bracketing values.
Spearman correlation adds a description of monotonicity.

A step is decisive when `step_r2 >= 0.05`, its improvement over the line is
at least 0.05, and it carries at least 80% of the curve's improvement over
the line. Otherwise, a curve is reported when `curve_r2 >= 0.10` and its
improvement over the line is at least 0.05. These are reporting thresholds,
not significance tests.

## Two examples

The first response jumps near a dose of four. The second follows a smooth
cubic curve. Each call below selects only the shape pass so the output
shows this comparison directly.


In [1]:
import numpy as np
import polars as pl
import onceover

rng = np.random.default_rng(11)
n = 1_200
dose = rng.uniform(0, 10, n)
df_step = pl.DataFrame({
    "dose": dose,
    "response": np.where(dose < 4, 1.0, 3.0) + rng.normal(0, 0.1, n),
})
level = rng.uniform(-1, 1, n)
df_curve = pl.DataFrame({"level": level, "response": level ** 3 + rng.normal(0, 0.05, n)})

shape_reports = {
    "Dose step": onceover.profile(df_step, target="response", detectors=["target_shapes"]),
    "Smooth cubic": onceover.profile(df_curve, target="response", detectors=["target_shapes"]),
}
shape_rows = []
for label, report in shape_reports.items():
    for finding in report.select(category="dependencies", target="response"):
        metrics = finding["evidence"][0]["metrics"]
        shape_rows.append({"Example": label, "Form": finding["specification"]["form"],
                           "Linear R²": metrics["linear_r2"], "Curve R²": metrics["curve_r2"],
                           "Step R²": metrics["step_r2"], "Claim": finding["claim"]})
assert {row["Form"] for row in shape_rows} == {"step_shape", "curved_shape"}
from great_tables import GT, loc, style
(GT(pl.DataFrame(shape_rows))
 .fmt_number(columns=["Linear R²", "Curve R²", "Step R²"], decimals=3)
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


Example,Form,Linear R²,Curve R²,Step R²,Claim
Dose step,step_shape,0.717,0.938,0.990,"dose relates to response mainly as a step at 4.001364: a single split explains 99.0% of response's variance, against 71.7% for a straight line; the step carries 123% of what a binned curve adds over the line"
Smooth cubic,curved_shape,0.827,0.974,0.532,"level relates to response as a monotone increasing curve, not a line: a binned curve explains 97.4% of response's variance, against 82.7% for a straight line (Spearman 0.95)"


In [2]:
import altair as alt

step_plot = (alt.Chart(df_step).mark_circle(size=13, opacity=0.25)
             .encode(x=alt.X("dose:Q", title="Dose"), y=alt.Y("response:Q", title="Response"))
             .properties(title="A threshold changes the level", width=330, height=230))
curve_plot = (alt.Chart(df_curve).mark_circle(size=13, opacity=0.25)
              .encode(x=alt.X("level:Q", title="Level"), y=alt.Y("response:Q", title="Response"))
              .properties(title="A smooth curve changes the slope", width=330, height=230))
(step_plot | curve_plot).interactive()


/home/hatch/workspace/onceover-venv-w314/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.HConcatChart(...)

## Reading and using a shape finding

The statement is a `dependencies/predictive` finding with form `step_shape`
or `curved_shape`. Read `linear_r2`, `curve_r2`, `step_r2`, `curvature`,
`step_share`, `spearman`, and `n_used` in its evidence. A step also records
its cut. Linear and unrelated predictors can produce no shape finding
because neither adds a decisive shape interpretation.

These scores describe the same observations used to choose the shape;
they are not out-of-sample predictive performance. The pass is univariate:
it does not adjust for other predictors or discover joint interactions.
A step-shaped pattern can also be a steep smooth transition at the
available resolution. The [discontinuity screen](step_discontinuities.ipynb)
asks a different question using a smooth background, split-half evaluation,
and local gap comparisons. The [interaction pass](interactions.ipynb)
examines joint effects under a fitted target model.


## The leakage screen

Designating a target also enables a second target-triggered pass, the `leakage` detector. Where this page's shape pass asks how a predictor relates to the target, the leakage screen asks whether any single column nearly determines the target on its own — the first check before modelling, because a column that restates the outcome, or was recorded after it, invalidates every model built on the frame while reading as a triumph.

Strength is measured per column on complete cases, on a 0–1 scale with a chance correction: for a categorical target, the share of the target's entropy the column accounts for beyond the fixed-margin permutation expectation (the reliable fraction of information); for a numeric target, Chatterjee's xi in the target direction for a numeric column, or the degrees-of-freedom-adjusted between-group share of variance for a categorical column. A column is stated as a `leakage_signature` finding when it determines the target exactly on the evaluated rows, or when its strength reaches 0.80 and is at least 1.5 times the runner-up's — a column out of line with the field, not a strong field. The claim names the column, its strength, and the runner-up, and the coverage entry reports the outcome either way. The statement names a signature to check; whether the column is in fact leakage depends on when it becomes known, which only the data's owner can say.


## The model-failure screen

Designating a target also enables a second target-triggered pass, the `model_failures` detector. Where this page's shape pass asks how each predictor relates to the target on its own, and the predictability screen asks how well the other columns jointly fit the target overall, the model-failure screen asks where that joint fit fails: a model can post a good overall score while its errors are large, or one-sided, inside one conjunction of conditions.

The screen fits the house GBM — the predictability screen's model, with the same configuration — by cross-fitting in 5 folds, so every modelled row's residual is out-of-fold: the row's prediction comes from the fold model that held it out. It then runs the slice search (the same condition space, split-sample confirmation, and Benjamini–Hochberg control as the slice-shift pass) over the residuals rather than over a column, and states a slice in one of two forms. The bias form states that the slice's mean residual differs materially from the model's overall mean residual — the model is one-sided there; for a binary target the statement gives the slice's event rate against its mean predicted probability, and the overall pair. The accuracy form states that the slice's mean absolute error is materially above the model's overall mean absolute error. Every statement names the slice, its size, its error, and the model's overall error.

If the cross-fitted model's pooled out-of-fold fit is at or near chance — R² at or below 0.01 for a numeric target, AUC at or below 0.55 for a binary target — the pass records that in its coverage entry and states nothing: where a model fails is not a meaningful question for a model that does not fit anywhere. Categorical targets with more than two classes are not evaluated; the coverage entry says so. The statement names a slice to investigate; whether the failure comes from a missing predictor, a changed relationship, or bad rows inside the slice is for the data's owner to judge.

Before fitting, the screen excludes every column that determines the target exactly — the leakage screen's determination test, shared with it — from the model inputs and from the slice variables, and its coverage entry names the excluded columns. Without the exclusion, a column that restates the target (one derived from the outcome, or the column a derived target was built from) hands the cross-fit a perfect score, and the statement budget is spent on slices of that column: within an all-events slice the error is one minus the mean prediction by construction, so the statements restate the leak instead of locating a failure. If the remaining model is at chance once the determining columns are out, the coverage-only outcome above is what the screen records.
